In [0]:
# ============================================================
# Silver — Source 16: CloudWatch Application Logs
# Regex extracts order_id from message text
# Source:  bronze.src_16_cloudwatch.app_logs
# Target:  silver.src_16_cloudwatch.app_logs
# ============================================================
from pyspark.sql import functions as F
from delta.tables import DeltaTable
BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_16_cloudwatch.app_logs'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_16_cloudwatch'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_16_cloudwatch')
print('Silver Source 16 CloudWatch — starting...')


In [0]:
bronze = spark.table(f'{BRONZE_CATALOG}.src_16_cloudwatch.app_logs')
total = bronze.count()
print(f'Bronze rows: {total}')

# Cast timestamp, normalise log_level
df = bronze \
    .withColumn('timestamp', F.to_timestamp(F.col('timestamp'))) \
    .withColumn('log_level', F.upper(F.trim(F.col('log_level'))))

# Regex extract order_id from message
# Covers: order #1234, order_1234, order 1234
df = df \
    .withColumn('order_id', F.regexp_extract(F.col('message'), r'order[_ #]+([0-9]+)', 1).cast('long')) \
    .withColumn('order_id', F.when(F.col('order_id') == 0, None).otherwise(F.col('order_id')))

# Bad rows — only reject null log_id or null timestamp
bad = df.filter(
    F.col('log_id').isNull() | F.col('timestamp').isNull()
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('app_logs'))

good = df.filter(
    F.col('log_id').isNotNull() & F.col('timestamp').isNotNull()
).dropDuplicates(['log_id'])

bad_count = bad.count()
good_count = good.count()
with_order = good.filter(F.col('order_id').isNotNull()).count()
print(f'Logs: {total} total → {good_count} clean, {bad_count} quarantined')
print(f'Logs with extracted order_id: {with_order}/{good_count} ({with_order/good_count*100:.1f}%)')

if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.log_id = s.log_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
print('Written')

if bad_count > 0:
    bad.select(
        F.lit('src_16_cloudwatch').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true').saveAsTable(QUARANTINE_TABLE)


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_16_cloudwatch.app_logs: {count} rows')
spark.sql(f'''
    SELECT log_level, COUNT(*) as cnt,
           SUM(CASE WHEN order_id IS NOT NULL THEN 1 ELSE 0 END) as with_order_id
    FROM {TARGET_TABLE}
    GROUP BY log_level ORDER BY cnt DESC
''').show()
